In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus']=False

import pandas as pd
import numpy as np
from sklearn import svm,metrics
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.linear_model import LinearRegression,LogisticRegression
from sklearn.preprocessing import PolynomialFeatures,StandardScaler  # 다항특성 자동생성, # 정규화클래스
from sklearn.linear_model import Ridge, Lasso
from sklearn.metrics import mean_absolute_error   # 평균절대값오차 오차범위
from sklearn.model_selection import train_test_split
from scipy.special import expit

In [81]:
# 2개만 분류 - 빙어/도미
# KNeighborClassifier - 다중분류
# 데이터 불러오기
fish = pd.read_csv('data/fish_logistic.csv')
fish.head(2)
fish['Species'].unique() # 7개의 종류(특성)을 가지고 있음

<StringArray>
['Bream', 'Roach', 'Whitefish', 'Parkki', 'Perch', 'Pike', 'Smelt']
Length: 7, dtype: str

In [82]:
# 자료 확인
fish.info()

<class 'pandas.DataFrame'>
RangeIndex: 159 entries, 0 to 158
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Species   159 non-null    str    
 1   Weight    159 non-null    float64
 2   Length    159 non-null    float64
 3   Diagonal  159 non-null    float64
 4   Height    159 non-null    float64
 5   Width     159 non-null    float64
dtypes: float64(5), str(1)
memory usage: 7.6 KB


In [83]:
fish.describe()

,Weight,Length,Diagonal,Height,Width
count,159.000000,159.000000,159.000000,159.000000,159.000000
mean,398.326415,28.415723,31.227044,8.970994,4.417486
std,357.978317,10.716328,11.610246,4.286208,1.685804
min,0.000000,8.400000,8.800000,1.728400,1.047600
25%,120.000000,21.000000,23.150000,5.944800,3.385650
50%,273.000000,27.300000,29.400000,7.786000,4.248500
75%,650.000000,35.500000,39.650000,12.365900,5.584500
max,1650.000000,63.400000,68.000000,18.957000,8.142000


In [84]:
data = fish[fish.columns[1:]].values
label = fish['Species'].values
label

<StringArray>
['Bream', 'Bream', 'Bream', 'Bream', 'Bream', 'Bream', 'Bream', 'Bream',
 'Bream', 'Bream',
 ...
 'Smelt', 'Smelt', 'Smelt', 'Smelt', 'Smelt', 'Smelt', 'Smelt', 'Smelt',
 'Smelt', 'Smelt']
Length: 159, dtype: str

In [85]:
# 학습데이터, 테스트데이터 분리
train_data,test_data,train_label,test_label = \
train_test_split(data,label)

In [86]:
# LogisticRegression, KNN은 무조건 정규화작업을 해야 함
ss = StandardScaler()
ss.fit(train_data)
train_scaled = ss.transform(train_data)
test_scaled = ss.transform(test_data)

In [105]:
##### 이진분류 형태 구성
##### Bream, Smelt생선만 분리 - 2개만 편의상 시행해 봄
bream_smelt_index = (train_label=='Bream')|(train_label=='Smelt') # Bream 또는 Smelt인 것만 뽑아라
train_bream_smelt = train_scaled[bream_smelt_index]
label_bream_smelt = train_label[bream_smelt_index]
label_bream_smelt.shape


(32,)

In [106]:
lr = LogisticRegression()
lr.fit(train_bream_smelt,label_bream_smelt)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following

In [107]:
print('분류:',lr.predict(train_bream_smelt[:-5]))
# 확률
print(lr.classes_)
print(lr.predict_proba(train_bream_smelt[:-5]))

분류: ['Bream' 'Bream' 'Smelt' 'Bream' 'Smelt' 'Bream' 'Smelt' 'Bream' 'Bream'
 'Bream' 'Bream' 'Smelt' 'Smelt' 'Bream' 'Bream' 'Smelt' 'Bream' 'Bream'
 'Bream' 'Smelt' 'Bream' 'Smelt' 'Smelt' 'Smelt' 'Bream' 'Bream' 'Bream']
['Bream' 'Smelt']
[[9.95876562e-01 4.12343782e-03]
 [9.80354780e-01 1.96452201e-02]
 [2.70568771e-02 9.72943123e-01]
 [9.93261962e-01 6.73803808e-03]
 [2.53950514e-02 9.74604949e-01]
 [9.97942985e-01 2.05701518e-03]
 [3.15409858e-02 9.68459014e-01]
 [9.81743239e-01 1.82567611e-02]
 [9.62478350e-01 3.75216505e-02]
 [9.95820033e-01 4.17996727e-03]
 [9.38243856e-01 6.17561438e-02]
 [2.72058688e-02 9.72794131e-01]
 [2.05351595e-02 9.79464840e-01]
 [9.99166167e-01 8.33833005e-04]
 [9.99234975e-01 7.65024797e-04]
 [2.18722555e-02 9.78127745e-01]
 [9.25480932e-01 7.45190676e-02]
 [9.92132135e-01 7.86786530e-03]
 [9.52378616e-01 4.76213837e-02]
 [2.56253384e-02 9.74374662e-01]
 [9.74082348e-01 2.59176523e-02]
 [6.06372906e-02 9.39362709e-01]
 [5.91870107e-02 9.40812989e-01]

In [108]:
# 'Weight', 'Length', 'Diagonal', 'Height', 'Width'
# 720,35,40.6,16.3618,6.09
# 7.5,10.5,11.6,1.972,1.16
# 생선의 무게,길이,대각선,높이,넓이를 입력받아, 어떤 생선인지 분류확률을 출력하시오.
# 분류생선확률 : Bream (99.76%)

In [109]:
in_data = input("입력하세요:")
in_data2 = in_data.split(',')
indata = list(map(float,in_data2))


In [111]:
data = fish[fish.columns[1:]].values
label = fish['Species'].values
label

<StringArray>
['Bream', 'Bream', 'Bream', 'Bream', 'Bream', 'Bream', 'Bream', 'Bream',
 'Bream', 'Bream',
 ...
 'Smelt', 'Smelt', 'Smelt', 'Smelt', 'Smelt', 'Smelt', 'Smelt', 'Smelt',
 'Smelt', 'Smelt']
Length: 159, dtype: str

In [ ]:
bream_smelt_index = (train_label =='bream')|(train_label=='Smelt')
train_bream_smelt = train_scaled[bream_smelt_index]
label_bream_smelt = train_label[bream_smelt_index]

In [112]:
in_data_scaled = ss.transform([in_data])
in_data_scaled

result = lr.predict(in_data_scaled)
result2 = lr.predict_proba(in_data_scaled)
print(lr.predict_proba(in_data_scaled))
if result[0] == 'Bream':
    result3 = result2[[0]]
else:
    result3 = result2[[1]]
print("test:", result3)
print(f"분류생선확률: {result[0]} ({result[1]})")

ValueError: could not convert string to float: '720,35,40.6,16.3618,6.09'

In [113]:
print(lr.coef_,lr.intercept_) # logisticRegression 선그래프 표현가능

[[-0.41624863 -0.57545579 -0.6580537  -1.05278035 -0.74790602]] [-1.71138429]


In [115]:
# z 의 값을 출력하는 함수
# z = a*무게 + b*길이 + c*대각선 + d*높이 + e*두께 + f : decision function
decisions = lr.decision_function(train_bream_smelt[:5])
decisions
# 음수면 bream

array([-5.48693608, -3.91008047,  3.58238441, -4.99322564,  3.64747788])

In [ ]:
# 이진분류에 사용되는 로지스틱함수(0~1까지 값으로 변경해서 보여줌)
from scipy.special import expit
print(expit(decisions))
# 낮으면 음, 높으면 양의 값을 가진다고 봄

[0.00412344 0.01964522 0.97294312 0.00673804 0.97460495]
